Question 1a

In [ ]:
import spacy

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")




def get_spacy_dependencies(sentence):
    """
    Uses spaCy to extract head-dependent relations
    which will act as the 'gold tree'.
    """
    doc = nlp(sentence)
    gold_deps = []

    for token in doc:
        if token.dep_ == "ROOT":
            gold_deps.append(("ROOT", token.text))
        else:
            gold_deps.append((token.head.text, token.text))

    return gold_deps




class DependencyParser:
    def __init__(self, sentence):
        self.stack = ['ROOT']
        self.buffer = sentence.split()
        self.dependencies = []

    def shift(self):
        if self.buffer:
            self.stack.append(self.buffer.pop(0))

    def left_arc(self):
        if len(self.stack) >= 2:
            head = self.stack[-1]
            dependent = self.stack[-2]
            self.dependencies.append((head, dependent))
            self.stack.pop(-2)

    def right_arc(self):
        if len(self.stack) >= 2:
            head = self.stack[-2]
            dependent = self.stack[-1]
            self.dependencies.append((head, dependent))
            self.stack.pop(-1)

def oracle(sentence, gold_dependencies):

    buffer = sentence.split()
    stack = ['ROOT']
    transitions = []

    while len(buffer) > 0 or len(stack) > 1:

        action_taken = False

        if len(stack) >= 2:

            s1 = stack[-1]
            s2 = stack[-2]

            # LEFT ARC
            if (s1, s2) in gold_dependencies:
                transitions.append("LEFT-ARC")
                stack.pop(-2)
                action_taken = True

            # RIGHT ARC
            elif (s2, s1) in gold_dependencies:

                has_dependents_in_buffer = any(
                    (s1, word) in gold_dependencies for word in buffer
                )

                if not has_dependents_in_buffer:
                    transitions.append("RIGHT-ARC")
                    stack.pop(-1)
                    action_taken = True

        # SHIFT
        if not action_taken:
            if len(buffer) > 0:
                transitions.append("SHIFT")
                stack.append(buffer.pop(0))
            else:
                break

    return transitions

sentence = "I parsed this sentence correctly"

print("Sentence:", sentence)

# Get dependencies automatically
gold_deps = get_spacy_dependencies(sentence)

print("\nspaCy Extracted Dependencies:")
for dep in gold_deps:
    print(f"{dep[0]} -> {dep[1]}")

# Generate transitions automatically
generated_transitions = oracle(sentence, gold_deps)

print("\nTransitions Generated by Oracle:")
print(generated_transitions)

# Run parser
parser = DependencyParser(sentence)

print("\nParser Execution:")
print(f"{'Stack':<35} | {'Buffer':<40} | {'Transition'}")
print("-" * 90)

for step in generated_transitions:

    if step == "SHIFT":
        parser.shift()

    elif step == "LEFT-ARC":
        parser.left_arc()

    elif step == "RIGHT-ARC":
        parser.right_arc()

    print(f"{str(parser.stack):<35} | {str(parser.buffer):<40} | {step}")

print("\nFinal Dependencies Built by Parser:")
for dep in parser.dependencies:
    print(dep)

Sentence: I parsed this sentence correctly

spaCy Extracted Dependencies:
parsed -> I
ROOT -> parsed
sentence -> this
parsed -> sentence
parsed -> correctly

Transitions Generated by Oracle:
['SHIFT', 'SHIFT', 'LEFT-ARC', 'SHIFT', 'SHIFT', 'LEFT-ARC', 'RIGHT-ARC', 'SHIFT', 'RIGHT-ARC', 'RIGHT-ARC']

Parser Execution:
Stack                               | Buffer                                   | Transition
------------------------------------------------------------------------------------------
['ROOT', 'I']                       | ['parsed', 'this', 'sentence', 'correctly'] | SHIFT
['ROOT', 'I', 'parsed']             | ['this', 'sentence', 'correctly']        | SHIFT
['ROOT', 'parsed']                  | ['this', 'sentence', 'correctly']        | LEFT-ARC
['ROOT', 'parsed', 'this']          | ['sentence', 'correctly']                | SHIFT
['ROOT', 'parsed', 'this', 'sentence'] | ['correctly']                            | SHIFT
['ROOT', 'parsed', 'sentence']      | ['correctly']   

In [ ]:
from spacy import displacy

def visualize_dependency_tree(sentence):

    doc = nlp(sentence)

    print("\nDependency Tree Visualization:")

    displacy.render(
        doc,
        style="dep",
        jupyter=True,
        options={
            "distance":120,
            "compact":False,
            "bg":"#ffffff",
            "color":"#000000"
        }
    )

# Run visualization
visualize_dependency_tree(sentence)


Dependency Tree Visualization:


2a

In [3]:
import torch
import torch.nn as nn

class OptimizedLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout_rate=0.2):
        super(OptimizedLSTM, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Adding dropout between LSTM layers to prevent overfitting
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout_rate if num_layers > 1 else 0)

        # Adding a fully connected layer to map LSTM outputs to the target task
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x shape: (batch_size, sequence_length, input_size)
        lstm_out, (hn, cn) = self.lstm(x)

        # Pass the output of the last time step through the linear layer
        # lstm_out[:, -1, :] gets the last hidden state of the sequence
        final_output = self.fc(lstm_out[:, -1, :])

        return final_output

# ==========================================
# Task 1: Time Series Prediction (e.g., Stock Prices, Weather)
# ==========================================
# Univariate time series usually has an input size of 1.
ts_input_size = 1
ts_hidden_size = 64     # Moderate hidden size is usually enough
ts_num_layers = 2       # 1 or 2 layers to capture temporal trends
ts_output_size = 1      # Predicting a single continuous value
ts_dropout = 0.2

time_series_model = OptimizedLSTM(ts_input_size, ts_hidden_size, ts_num_layers, ts_output_size, ts_dropout)

# ==========================================
# Task 2: Text Generation (Character or Word level)
# ==========================================
# Input size represents the word/char embedding dimension.
text_input_size = 256
text_hidden_size = 512  # Larger hidden size needed for complex language patterns
text_num_layers = 3     # Deeper network for hierarchical language structures
text_output_size = 10000 # The size of your vocabulary (predicting the next word)
text_dropout = 0.5      # Higher dropout needed due to large parameter count

text_gen_model = OptimizedLSTM(text_input_size, text_hidden_size, text_num_layers, text_output_size, text_dropout)
print(time_series_model)
print(text_gen_model)

OptimizedLSTM(
  (lstm): LSTM(1, 64, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)
OptimizedLSTM(
  (lstm): LSTM(256, 512, num_layers=3, batch_first=True, dropout=0.5)
  (fc): Linear(in_features=512, out_features=10000, bias=True)
)


Problem 2c

In [5]:
import torch
import torch.nn as nn

class CustomGRU(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(CustomGRU, self).__init__()

        self.hidden_size = hidden_size

        # Linear layer to compute Update gate (z_t) and Reset gate (r_t)
        # [x_t , h_(t-1)] → 2 * hidden_size
        self.gate_linear = nn.Linear(input_size + hidden_size, 2 * hidden_size)

        # Linear layer for candidate hidden state
        self.candidate_linear = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(self, x, h_prev):

        # Combine input and previous hidden state
        combined = torch.cat((x, h_prev), dim=1)

        # -------------------------------------------------
        # Update Gate and Reset Gate
        # z_t = σ(Wz[x_t , h_(t-1)])
        # r_t = σ(Wr[x_t , h_(t-1)])
        # -------------------------------------------------
        gates = torch.sigmoid(self.gate_linear(combined))
        z_t, r_t = gates.chunk(2, dim=1)

        # -------------------------------------------------
        # Candidate hidden state
        # h~_t = tanh(Wh[x_t , r_t * h_(t-1)])
        # -------------------------------------------------
        combined_reset = torch.cat((x, r_t * h_prev), dim=1)
        h_candidate = torch.tanh(self.candidate_linear(combined_reset))

        # -------------------------------------------------
        # Final hidden state
        # h_t = (1 - z_t) * h_(t-1) + z_t * h~_t
        # -------------------------------------------------
        h_next = (1 - z_t) * h_prev + z_t * h_candidate

        return h_next


# Example parameters
input_size = 10
hidden_size = 50

# Create GRU cell
gru_cell = CustomGRU(input_size, hidden_size)

# Dummy input to test
sample_input = torch.randn(1, input_size)
sample_hidden = torch.randn(1, hidden_size)

output = gru_cell(sample_input, sample_hidden)

print("Output hidden state shape:", output.shape)

Output hidden state shape: torch.Size([1, 50])
